# ⬇️ InnerUpload — Colab Downloader

دانلود فایل از هر لینکی روی سرورهای Google و ذخیره مستقیم در Google Drive شما.

Download files from any URL using Google's servers and save directly to your Google Drive.

---

**پشتیبانی از / Supports:**
- 🔗 لینک مستقیم HTTP/HTTPS
- ☁️ Google Drive (لینک share)
- 📦 Dropbox
- 🐙 GitHub Releases
- 🌐 اکثر سرورهای عمومی

**نحوه استفاده / How to use:**
1. روی `Runtime → Run all` کلیک کنید
2. اجازه دسترسی به Google Drive را بدهید
3. لینک(ها) را وارد کنید و دانلود را شروع کنید

In [ ]:
#@title ⚙️ Step 1 — Install dependencies (run once)
import subprocess, sys

packages = ["gdown", "tqdm", "requests"]

print("📦 Installing dependencies...")
for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", pkg])
print("✅ Done!")

In [ ]:
#@title 📂 Step 2 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive mounted at /content/drive")

In [ ]:
#@title ⚙️ Step 3 — Settings

#@markdown ### 📁 Save Location
#@markdown نام پوشه‌ای که فایل‌ها در Google Drive شما ذخیره می‌شوند
DRIVE_FOLDER_NAME = "InnerUpload"  #@param {type:"string"}

#@markdown ---
#@markdown ### 🔗 Download Links
#@markdown هر لینک را در یک خط جداگانه وارد کنید / One URL per line
DOWNLOAD_LINKS = ""  #@param {type:"string"}

#@markdown ---
#@markdown ### ⚙️ Options
OVERWRITE_IF_EXISTS = False  #@param {type:"boolean"}

# ── Setup save directory ──────────────────────────────────────────────────────
from pathlib import Path

SAVE_DIR = Path(f"/content/drive/MyDrive/{DRIVE_FOLDER_NAME}")
SAVE_DIR.mkdir(parents=True, exist_ok=True)
TEMP_DIR = Path("/content/innerupload_temp")
TEMP_DIR.mkdir(parents=True, exist_ok=True)

print(f"✅ Files will be saved to: {SAVE_DIR}")

In [ ]:
# ─── Core download logic ───────────────────────────────────────────────────────
import re
import os
import time
import shutil
import requests
import mimetypes
import gdown
from pathlib import Path
from tqdm.notebook import tqdm
from urllib.parse import urlparse, parse_qs, unquote


# ── URL normalizers ────────────────────────────────────────────────────────────

def normalize_google_drive(url: str) -> str:
    """
    Convert any Google Drive share URL to a direct gdown-compatible URL.
    Handles: /file/d/ID/view, /open?id=ID, /uc?id=ID
    """
    # Extract file ID from various GDrive URL formats
    patterns = [
        r"/file/d/([a-zA-Z0-9_-]+)",
        r"[?&]id=([a-zA-Z0-9_-]+)",
        r"/d/([a-zA-Z0-9_-]+)",
    ]
    for pat in patterns:
        m = re.search(pat, url)
        if m:
            file_id = m.group(1)
            return f"https://drive.google.com/uc?id={file_id}&export=download"
    return url  # return as-is if we can't parse it


def normalize_dropbox(url: str) -> str:
    """Force Dropbox direct download by setting dl=1."""
    url = re.sub(r"[?&]dl=0", "", url)
    sep = "&" if "?" in url else "?"
    return url + sep + "dl=1"


def detect_and_normalize(url: str) -> tuple[str, str]:
    """
    Detect URL type, normalize it, and return (normalized_url, url_type).
    url_type: 'gdrive' | 'direct'
    """
    host = urlparse(url).netloc.lower()

    if "drive.google.com" in host or "docs.google.com" in host:
        return normalize_google_drive(url), "gdrive"

    if "dropbox.com" in host:
        return normalize_dropbox(url), "direct"

    return url, "direct"


# ── Filename helpers ───────────────────────────────────────────────────────────

def get_filename_from_response(response: requests.Response, url: str) -> str:
    """Try to extract a clean filename from Content-Disposition or URL."""
    cd = response.headers.get("Content-Disposition", "")
    m = re.search(r'filename\*?=(?:UTF-8\'\')?"?([^"\n;]+)"?', cd, re.IGNORECASE)
    if m:
        return unquote(m.group(1).strip())

    # Fall back to URL path
    path = unquote(urlparse(url).path)
    name = path.rstrip("/").split("/")[-1]
    if name and "." in name:
        return name

    # Last resort: use content-type to guess extension
    ct = response.headers.get("Content-Type", "").split(";")[0].strip()
    ext = mimetypes.guess_extension(ct) or ""
    return f"file_{int(time.time())}{ext}"


# ── Downloaders ────────────────────────────────────────────────────────────────

def download_direct(url: str, dest_dir: Path, overwrite: bool = False) -> Path:
    """
    Download any direct HTTP/HTTPS URL with a progress bar.
    Streams the response to avoid RAM issues with large files.
    """
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0.0.0 Safari/537.36"
        )
    }

    session = requests.Session()

    # Follow redirects and get final response
    response = session.get(url, headers=headers, stream=True, timeout=30, allow_redirects=True)
    response.raise_for_status()

    filename = get_filename_from_response(response, response.url)
    dest_path = dest_dir / filename

    if dest_path.exists() and not overwrite:
        print(f"   ⏭️  File already exists, skipping: {filename}")
        return dest_path

    total = int(response.headers.get("Content-Length", 0))

    with open(dest_path, "wb") as f, tqdm(
        total=total,
        unit="B",
        unit_scale=True,
        unit_divisor=1024,
        desc=filename[:50],
        dynamic_ncols=True,
    ) as bar:
        for chunk in response.iter_content(chunk_size=1024 * 1024):  # 1 MB chunks
            if chunk:
                f.write(chunk)
                bar.update(len(chunk))

    return dest_path


def download_gdrive(url: str, dest_dir: Path, overwrite: bool = False) -> Path:
    """
    Download a Google Drive file using gdown (handles large-file confirmation).
    """
    # gdown can write to a folder directly
    output = str(dest_dir) + "/"  # trailing slash = save into folder
    result = gdown.download(url, output=output, quiet=False, fuzzy=True)
    if result is None:
        raise RuntimeError("gdown returned None — the file may be private or the link is invalid.")
    return Path(result)


# ── Main download dispatcher ───────────────────────────────────────────────────

def download(url: str, dest_dir: Path, overwrite: bool = False) -> Path:
    """
    Detect URL type and dispatch to the right downloader.
    Returns the path of the downloaded file.
    """
    url = url.strip()
    if not url:
        raise ValueError("Empty URL")

    normalized_url, url_type = detect_and_normalize(url)

    if url_type == "gdrive":
        return download_gdrive(normalized_url, dest_dir, overwrite)
    else:
        return download_direct(normalized_url, dest_dir, overwrite)


print("✅ Download engine loaded.")

In [ ]:
#@title ▶️ Step 4 — Start Download

from google.colab import auth as _colab_auth
from googleapiclient.discovery import build as _build

# Authenticate with Google (reuses existing Colab session — no extra popup if already authed)
_colab_auth.authenticate_user()
_drive_svc = _build('drive', 'v3', cache_discovery=False)


def get_save_folder_id(service, folder_name: str) -> str | None:
    """
    Look up the Drive folder ID by name.
    Used to scope file searches so same-name files in other folders don't interfere.
    """
    safe_name = folder_name.replace("'", "\\'")
    results = service.files().list(
        q=(
            f"name='{safe_name}'"
            " and mimeType='application/vnd.google-apps.folder'"
            " and trashed=false"
        ),
        fields="files(id)",
        pageSize=1,
        orderBy="createdTime desc",
    ).execute()
    files = results.get('files', [])
    return files[0]['id'] if files else None


def get_drive_share_link(service, filename: str, folder_id: str | None) -> str | None:
    """
    Find a file in Drive by name, scoped to the specific save folder.
    Scoping to folder_id avoids picking the wrong file when the same
    filename exists elsewhere in Drive.
    Grants public read access and returns a shareable link.
    """
    safe_name = filename.replace("'", "\\'")
    parent_filter = f" and '{folder_id}' in parents" if folder_id else ""
    results = service.files().list(
        q=f"name='{safe_name}' and trashed=false{parent_filter}",
        fields="files(id)",
        pageSize=1,
        orderBy="createdTime desc",
    ).execute()
    files = results.get('files', [])
    if not files:
        return None
    file_id = files[0]['id']
    service.permissions().create(
        fileId=file_id,
        body={'type': 'anyone', 'role': 'reader'},
    ).execute()
    return f"https://drive.google.com/file/d/{file_id}/view?usp=sharing"


# Resolve folder ID once upfront so every file lookup is scoped correctly
_folder_id = get_save_folder_id(_drive_svc, DRIVE_FOLDER_NAME)
if _folder_id:
    print(f"📂 Save folder found (id: {_folder_id})")
else:
    print("⚠️  Save folder not found in Drive — links will still work but search is unscoped")


# ── Parse links ────────────────────────────────────────────────────────────────
links = [l.strip() for l in DOWNLOAD_LINKS.strip().splitlines() if l.strip()]

if not links:
    print("⚠️  No links found. Please enter at least one URL in Step 3.")
else:
    print(f"📋 {len(links)} link(s) queued\n")
    print("=" * 60)

    success, failed = [], []

    for i, url in enumerate(links, 1):
        print(f"\n[{i}/{len(links)}] 🔗 {url[:80]}{'...' if len(url) > 80 else ''}")
        try:
            # Download to temp dir first, then move to Drive
            temp_path = download(url, TEMP_DIR, overwrite=OVERWRITE_IF_EXISTS)

            final_path = SAVE_DIR / temp_path.name
            if final_path.exists() and not OVERWRITE_IF_EXISTS:
                print(f"   ⏭️  Already in Drive: {temp_path.name}")
                temp_path.unlink(missing_ok=True)
            else:
                print(f"   📤 Moving to Drive...")
                shutil.move(str(temp_path), str(final_path))
                size_mb = final_path.stat().st_size / (1024 * 1024)
                print(f"   ✅ Saved: {final_path.name} ({size_mb:.1f} MB)")

            # ── Get shareable Drive link (scoped to save folder) ──────────────
            share_link = get_drive_share_link(_drive_svc, final_path.name, _folder_id)
            if share_link:
                print(f"   🔗 Drive link: {share_link}")
            else:
                print(f"   ⚠️  Could not retrieve Drive link (file saved successfully)")

            success.append((url, share_link))

        except Exception as e:
            print(f"   ❌ Failed: {e}")
            failed.append((url, str(e)))
            # Clean up any broken temp file
            for f in TEMP_DIR.iterdir():
                f.unlink(missing_ok=True)

    # ── Summary ────────────────────────────────────────────────────────────────
    print("\n" + "=" * 60)
    print(f"📊 Summary: {len(success)} succeeded, {len(failed)} failed")
    print(f"📁 Saved to: {SAVE_DIR}")

    if success:
        print("\n✅ Drive links:")
        for src_url, drive_link in success:
            label = src_url[:60] + ('...' if len(src_url) > 60 else '')
            link_str = drive_link if drive_link else '(link unavailable)'
            print(f"   • {label}")
            print(f"     → {link_str}")

    if failed:
        print("\n❌ Failed links:")
        for url, err in failed:
            print(f"   • {url[:70]}")
            print(f"     → {err}")

In [ ]:
#@title 📋 Bonus — Download from a text file of links
#@markdown اگه لینک‌ها رو داخل یه فایل txt دارید، اینجا آدرس اون فایل رو بدید
#@markdown Upload your .txt file (one URL per line) and enter its path below.

LINKS_FILE_PATH = "/content/links.txt"  #@param {type:"string"}

from pathlib import Path

links_file = Path(LINKS_FILE_PATH)
if not links_file.exists():
    print(f"⚠️  File not found: {LINKS_FILE_PATH}")
    print("   Upload your file via the Files panel on the left sidebar.")
else:
    file_links = [l.strip() for l in links_file.read_text().splitlines() if l.strip() and not l.startswith("#")]
    print(f"📋 Found {len(file_links)} link(s) in file\n")
    print("=" * 60)

    success_f, failed_f = [], []

    for i, url in enumerate(file_links, 1):
        print(f"\n[{i}/{len(file_links)}] 🔗 {url[:80]}{'...' if len(url) > 80 else ''}")
        try:
            temp_path = download(url, TEMP_DIR, overwrite=OVERWRITE_IF_EXISTS)
            final_path = SAVE_DIR / temp_path.name
            if final_path.exists() and not OVERWRITE_IF_EXISTS:
                print(f"   ⏭️  Already in Drive: {temp_path.name}")
                temp_path.unlink(missing_ok=True)
            else:
                print(f"   📤 Moving to Drive...")
                shutil.move(str(temp_path), str(final_path))
                size_mb = final_path.stat().st_size / (1024 * 1024)
                print(f"   ✅ Saved: {final_path.name} ({size_mb:.1f} MB)")
            success_f.append(url)
        except Exception as e:
            print(f"   ❌ Failed: {e}")
            failed_f.append((url, str(e)))
            for f in TEMP_DIR.iterdir():
                f.unlink(missing_ok=True)

    print("\n" + "=" * 60)
    print(f"📊 Summary: {len(success_f)} succeeded, {len(failed_f)} failed")
    print(f"📁 Saved to: {SAVE_DIR}")